In [3]:
#checking the content length of the retrieved scrapper

import json
from pathlib import Path
import re
import sys

#Diagnosting paths first
# notebook_dir = Path('.').resolve()
# print(f"Current working directory: {notebook_dir}")

#Add project root path so we can import src modules
sys.path.append(str(Path('..').resolve()))

#Load all scraped docs
docs = [
    json.loads(f.read_text())
    for f in Path('../data/raw').glob('*.json')
    ]

print(f"Loaded {len(docs)} documents")
for doc in docs:
    print(f" {doc['source']} | {doc['section']} | {doc['content_length']} chars | {doc['title'][:50]}")

Loaded 7 documents
 mparticle | general | 21392 chars | Introduction
 aep | profile | 12028 chars | Real-Time Customer Profile Overview | Adobe Experi
 aep | segmentation | 10154 chars | Segmentation Service Overview | Adobe Experience P
 mparticle | general | 13621 chars | Create an Input
 aep | governance | 6812 chars | Data Governance Overview | Adobe Experience Platfo
 aep | schema | 13432 chars | XDM System Overview | Adobe Experience Platform
 aep | identity | 10595 chars | Identity Service Overview | Adobe Experience Platf


In [5]:
# Analyzing raw content structure. 
 # Test on each source type - AEP, mParticle behave differently
for i, doc in enumerate(docs):
    content = doc['content']

    heading_markers = content.count('\n## ')
    double_newlines = content.count('\n\n')
    single_newlines = content.count('\n')

    print(f"Doc {i}: {doc['source']:10} | {doc['section']:12} | "
          f"## markers: {heading_markers:3} | " 
    f"\\n\\n: {double_newlines:4} | "      
    f"\\n: {single_newlines:4} | "  
    f"chars: {len(content):6}")

print("\n=== FIRST 500 CHARS OF DOC 0 (raw) ===")
print(repr(docs[0]['content'][:500]))

print("\n=== FIRST 500 CHARS OF DOC 1 (raw) ===")
print(repr(docs[1]['content'][:500]))

Doc 0: mparticle  | general      | ## markers:   8 | \n\n:   16 | \n:  331 | chars:  21392
Doc 1: aep        | profile      | ## markers:  17 | \n\n:   33 | \n:   86 | chars:  12028
Doc 2: aep        | segmentation | ## markers:  17 | \n\n:   33 | \n:   96 | chars:  10154
Doc 3: mparticle  | general      | ## markers:   8 | \n\n:   16 | \n:  294 | chars:  13621
Doc 4: aep        | governance   | ## markers:   7 | \n\n:   13 | \n:   58 | chars:   6812
Doc 5: aep        | schema       | ## markers:  12 | \n\n:   24 | \n:   93 | chars:  13432
Doc 6: aep        | identity     | ## markers:   6 | \n\n:   11 | \n:   91 | chars:  10595

=== FIRST 500 CHARS OF DOC 0 (raw) ===
'Data Subject Request API Version 1 and 2\nData Subject Request API Version 3\nPlatform API Overview\nCalculated Attributes\nField Transformations\nWarehouse Sync API Overview\nWarehouse Sync API Tutorial\nWarehouse Sync API Reference\nWarehouse Sync SQL Reference\nWarehouse Sync Troubleshooting Guide\nWarehouse Sync API 

In [4]:
# Deep dive on content strategy
# check where first real heading appears in each doc

print("=== FIRST HEADING POSITION===\n")

for i, doc in enumerate(docs):
    content = doc['content']
    first_heading_pos = content.find('\n## ') # returns the line numbers

    #content before the first heading (navigation noise)
    preamble = content[:first_heading_pos].strip() if first_heading_pos > 0 else ""
    preamble_lines = len(preamble.split('\n')) if preamble else 0 

    print(f"oc {i}: {doc['source']:10} | {doc['section']:12}) | " 
          f"First ## at char: {first_heading_pos:5} | "
          f"preamble_lines: {preamble_lines:3}")
    
    if preamble_lines > 5:
        print( "Navigation noise detected: first 100 chars of preamble:")
        print(f"  {repr(preamble[:100])}")
    print()


#Show what a clean AEP section looks like vs mParticle

print("===AEP section sample(Doc 1)===")
aep_doc = docs[1]
sections_aep = aep_doc['content'].split('\n## ')
print(f"Section count: {len(sections_aep)}")

for s in sections_aep[1:4]: # skip preamble show 3 sections
    lines = s.strip().split('\n')
    heading = lines[0] if lines else ""
    body_preview = ' '.join(lines[1:])[:200] if len(lines) > 1 else ""
    print(f"\n ## {heading}")
    print(f" {body_preview}")


#mParticle section
print("===mParticle section sample(Doc 0)===")
mp_doc = docs[0]
sections_mp = mp_doc['content'].split('\n## ')
print(f"Section count: {len(sections_mp)}")

for s in sections_mp[1:4]: # skip preamble show 3 sections
    lines = s.strip().split('\n')
    heading = lines[0] if lines else ""
    body_preview = ' '.join(lines[1:])[:200] if len(lines) > 1 else ""
    print(f"\n ## {heading}")
    print(f" {body_preview}")


=== FIRST HEADING POSITION===

oc 0: mparticle  | general     ) | First ## at char:  7750 | preamble_lines: 224
Navigation noise detected: first 100 chars of preamble:
  'Data Subject Request API Version 1 and 2\nData Subject Request API Version 3\nPlatform API Overview\nCa'

oc 1: aep        | profile     ) | First ## at char:     0 | preamble_lines:   0

oc 2: aep        | segmentation) | First ## at char:     0 | preamble_lines:   0

oc 3: mparticle  | general     ) | First ## at char:  8467 | preamble_lines: 231
Navigation noise detected: first 100 chars of preamble:
  'Data Subject Request API Version 1 and 2\nData Subject Request API Version 3\nPlatform API Overview\nCa'

oc 4: aep        | governance  ) | First ## at char:     0 | preamble_lines:   0

oc 5: aep        | schema      ) | First ## at char:  1217 | preamble_lines:   4

oc 6: aep        | identity    ) | First ## at char:     0 | preamble_lines:   0

===AEP section sample(Doc 1)===
Section count: 18

 ## Real-Time Cu

In [6]:
# Experiment heading-based chunking. 
def strip_preamble(content: str) -> str:
    """remove anavigation noise before first real heading.
    mParticle docs have a link list before ###markers
    AEP docs start cleanly but this is safe for both
    """
    first_heading = content.find('\n## ')
    if first_heading > 0 :
        return content[first_heading:].strip()
    return content.strip()

def split_into_sections(content:str)-> str:
    """split by ## headings into (heading, body) pairs"""
    cleaned = strip_preamble(content)
    parts = cleaned.split('\n##')

    sections = []
    for part in parts:
        if not part.strip():
            continue
        lines = part.strip().split('\n',1)
        heading = lines[0].strip()
        body = lines[1].strip() if len(lines) > 1 else " "
        if body and len(body) > 50:
            sections.append((heading,body))
    return sections

#Test on all docs
print("===SECTION QUALITY ACROSS ALL DOCS==\n")


for i, doc in enumerate(docs):
    sections = split_into_sections(doc['content'])
    avg_section_len = sum(len(b) for _, b in sections) // max(len(sections),1)

    print(f"Doc {i}: {doc['source']:10} | {doc['section']:12} | "
          f"sections: {len(sections):3} | "
          f"Avg section: {avg_section_len:5} chars")
    
    #show first 2 sections
    for heading, body in sections[:2]:
        print(f" [{heading[:50]}] -> {len(body)} chars")
        print(f" {body[:100]}...")
    print()

===SECTION QUALITY ACROSS ALL DOCS==

Doc 0: mparticle  | general      | sections:   8 | Avg section:  1663 chars
 [## Accounts, organizations, and workspaces] -> 4005 chars
 mParticle creates a unique organization for you. It’s the container for all data and metadata relate...
 [Using organizations, accounts, and workspaces] -> 630 chars
 Use organizations, accounts, and workspaces to manage multiple brands, regions, and to manage custom...

Doc 1: aep        | profile      | sections:  17 | Avg section:   666 chars
 [## Real-Time Customer Profile overview] -> 608 chars
 Adobe Experience Platform enables you to drive coordinated, consistent, and relevant experiences for...
 [Profile in Experience Platform] -> 138 chars
 The relationship between Real-Time Customer Profile and other services within Experience Platform is...

Doc 2: aep        | segmentation | sections:  17 | Avg section:   560 chars
 [## Segmentation Service overview] -> 453 chars
 Adobe Experience Platform Segmentation

In [7]:
# test chunk sizes within sections
# Key question - Are sections small enough to be single chunks, or do they need further splitting

CHILD_SIZE = 1024 #chars ~256 tokens
CHILD_OVERLAP = 128 #chars ~32 tokens


def section_to_chunks(body:str, heading:str)-> list[dict]:
    """Split one section body into child chunks with overlap"""
    chunks = []
    start = 0 
    idx = 0

    while start < len(body):
        end = start + CHILD_SIZE

        #Try to end at sentence boundary
        if end < len(body):
            boundary = body.rfind('.',start, end)
            if boundary > start + (CHILD_SIZE//2):
                end = boundary + 1 

        chunk_text = body[start:end].strip()
        if len(chunk_text)> 50:
            chunks.append ({
                "index": idx,
                "heading": heading, 
                "chars" : len(chunk_text),
                "token_est": len(chunk_text)//4,
                "preview_start":chunk_text[:80],
                "preview_end": chunk_text[-80:],
                "ends_at_sentence": chunk_text.rstrip().endswith(('.','?','!'))                                          
                })
        start = end = CHILD_OVERLAP
        idx += 1
    return chunks

print("=== CHUNK DISTRIBUTION PER DOC==\n")

total_chunks_all = 0

for i, doc in enumerate(docs):
    sections = split_into_sections(doc['content'])
    doc_chunks = []

    for heading, body in sections:
        chunks = section_to_chunks(body, heading)
        doc_chunks.extend(chunks)

    total_chunks_all += len(doc_chunks)
    sizes = [c['tokens_est'] for c in doc_chunks]

    print(f"Doc {i}: {doc['spurce']:10} | {doc['section']:12} | "
          f"Chunks: {len(doc_chunks):3} | "
          f"Avg tokens: {sum(sizes) // max(len(sizes),1):4} | "
          f"Min: {min(sizes):3} | Max: {max(sizes):3}")
    
print(f"\n Total chunks across all docs: {total_chunks_all}")
print(f"Estimated embedding cost:"
      f"~${total_chunks_all * 1024 * 0.00002 / 1000:.4f} USD")


=== CHUNK DISTRIBUTION PER DOC==



KeyboardInterrupt: 

In [1]:
# ── Cell 6: Visual quality check — the most important cell ───────────────────
# Read these chunks out loud mentally.
# Ask: "If this was returned by a search engine, would it answer the question?"

print("=== CHUNK QUALITY INSPECTION ===\n")
print("For each chunk below, ask:")
print("1. Does it make sense in isolation?")
print("2. Does it start and end cleanly?")
print("3. Would it answer a user question about CDP?\n")

# Sample from each source
for source in ['aep', 'mparticle']:
    source_docs = [d for d in docs if d['source'] == source]
    if not source_docs:
        continue
    
    doc = source_docs[0]
    sections = split_into_sections(doc['content'])
    
    print(f"{'='*60}")
    print(f"SOURCE: {source.upper()} | Doc: {doc['title'][:50]}")
    print(f"{'='*60}")
    
    # Show chunks from first 2 sections
    for heading, body in sections[:2]:
        chunks = section_to_chunks(body, heading)
        print(f"\n  Section: [{heading}]")
        print(f"  Body length: {len(body)} chars → {len(chunks)} chunk(s)")
        
        for chunk in chunks[:2]:  # Max 2 chunks per section
            print(f"\n  ── Chunk {chunk['index']+1} "
                  f"({chunk['tokens_est']} tokens) ──")
            print(f"  START: {chunk['preview_start']}...")
            print(f"  END:   ...{chunk['preview_end']}")
            print(f"  Clean sentence end: {chunk['ends_at_sentence']}")

=== CHUNK QUALITY INSPECTION ===

For each chunk below, ask:
1. Does it make sense in isolation?
2. Does it start and end cleanly?
3. Would it answer a user question about CDP?



NameError: name 'docs' is not defined

In [ ]:
# ── Cell 7: Overlap verification ──────────────────────────────────────────────
# Verify overlap is working — chunk N+1 should start
# before chunk N ends, sharing some content

print("=== OVERLAP VERIFICATION ===\n")

# Pick a section long enough to have multiple chunks
for doc in docs:
    sections = split_into_sections(doc['content'])
    long_sections = [(h, b) for h, b in sections if len(b) > CHILD_SIZE * 1.5]
    
    if long_sections:
        heading, body = long_sections[0]
        chunks = section_to_chunks(body, heading)
        
        if len(chunks) >= 2:
            print(f"Doc: {doc['title'][:50]}")
            print(f"Section: [{heading}]")
            print(f"Total chunks in section: {len(chunks)}\n")
            
            print(f"Chunk 1 ends with:")
            print(f"  ...{chunks[0]['preview_end']}")
            print(f"\nChunk 2 starts with:")
            print(f"  {chunks[1]['preview_start']}...")
            print(f"\nOverlap visible: "
                  f"{'YES' if chunks[0]['preview_end'][:20] in body[body.find(chunks[1]['preview_start'][:20])-50:body.find(chunks[1]['preview_start'][:20])+50] else 'CHECK MANUALLY'}")
            break

In [ ]:
# ── Cell 8: Final comparison and decision ─────────────────────────────────────

print("=== CHUNKING STRATEGY VERDICT ===\n")

# Run both strategies and compare
all_heading_chunks = []
for doc in docs:
    sections = split_into_sections(doc['content'])
    for heading, body in sections:
        chunks = section_to_chunks(body, heading)
        all_heading_chunks.extend(chunks)

# Simple recursive for comparison
def simple_recursive(text, size=1024, overlap=128):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if len(c) > 50]

all_recursive_chunks = []
for doc in docs:
    cleaned = strip_preamble(doc['content'])
    all_recursive_chunks.extend(simple_recursive(cleaned))

heading_sizes = [c['tokens_est'] for c in all_heading_chunks]
recursive_sizes = [len(c)//4 for c in all_recursive_chunks]

print(f"{'Metric':<30} {'Heading-Based':>15} {'Recursive':>12}")
print("-" * 58)
print(f"{'Total chunks':<30} {len(all_heading_chunks):>15} {len(all_recursive_chunks):>12}")
print(f"{'Avg tokens':<30} {sum(heading_sizes)//len(heading_sizes):>15} {sum(recursive_sizes)//len(recursive_sizes):>12}")
print(f"{'Min tokens':<30} {min(heading_sizes):>15} {min(recursive_sizes):>12}")
print(f"{'Max tokens':<30} {max(heading_sizes):>15} {max(recursive_sizes):>12}")
print(f"{'Est. embedding cost USD':<30} ${len(all_heading_chunks)*1024*0.00002/1000:>14.4f} ${len(all_recursive_chunks)*1024*0.00002/1000:>11.4f}")

print(f"""
Based on your data:
- Both AEP and mParticle have real ## heading structure
- mParticle preamble noise is cleanly strippable
- Heading-based preserves semantic boundaries

RECOMMENDATION: Heading-based chunking with preamble stripping.

The heading boundaries in your docs are real content boundaries —
XDM Overview, Identity Resolution, Segmentation are genuinely
different topics that should never be merged in a single chunk.
Recursive splitting would sometimes merge them.

Next step: implement chunker.py using split_into_sections()
and section_to_chunks() as the core functions — they're
already tested and working in this notebook.
""")